# General Feature-space Cloaking - Caltech-101 Full Final Benchmark

Notebook final cho Cloaking. Muc tieu khong phai train victim classifier, ma do muc do feature extractor bi keo lech sau khi them nhieu adversarial bang PGD.

- Dataset: Caltech-101 resized 224x224
- Surrogate/feature extractor: ResNet-50 pretrained ImageNet
- Objective: keo feature anh protected ve target feature duoc chon bang `max_dist`
- Metric chinh: cosine(original, protected), feature L2 shift, target cosine gain
- Metric chat luong anh: PSNR, SSIM, L-infinity


## 1. Setup Colab / GitHub repo


In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
REPO_DIR = Path("/content/adversarial-data-protection")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from src.datasets import get_caltech101
from src.evaluation import compute_linf, compute_psnr, compute_ssim
from src.models import get_surrogate_resnet50
from src.techniques.cloaking import cloak_images, select_cloak_target
from src.pipeline import tensor_to_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 2. Google Drive paths and config


## Full run co y nghia gi?

Voi Cloaking, full dataset khong tao ra benchmark train/test nhu Unlearnable. No co y nghia thong ke: thay vi chung minh tren vai anh demo, ta do trung binh tren toan bo Caltech-101 de ket qua feature shift on dinh va dang tin hon.

Neu can hoan thanh nhanh, doi `SUBSET_SIZE = 2000`. Neu can so lieu final day du, giu `SUBSET_SIZE = None`.


In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"
DRIVE_DATA_ROOT = None

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_DATA_ROOT = Path(f"{DRIVE_PROJECT_DIR}/data")
    RESULTS_ROOT = f"{DRIVE_PROJECT_DIR}/results"
    # Read images from Colab local disk, not directly from Drive.
    # Direct Drive reads can fail with: Transport endpoint is not connected.
    DATA_ROOT = "/content/data"
    Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
    Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

    drive_caltech = DRIVE_DATA_ROOT / "caltech101"
    local_caltech = Path(DATA_ROOT) / "caltech101"
    if drive_caltech.exists() and not local_caltech.exists():
        print("Copying Caltech-101 from Drive to local Colab disk. This is slower once, then faster/stabler during training.")
        shutil.copytree(drive_caltech, local_caltech, dirs_exist_ok=True)
    elif local_caltech.exists():
        print("Using existing local Caltech-101 cache:", local_caltech)
    else:
        print("Drive Caltech-101 cache not found. torchvision will download Caltech-101 into local /content/data.")

SEED = 42
IMG_SIZE = 224
# Final run: None means use all available Caltech-101 images.
# If Colab runtime is tight, set SUBSET_SIZE = 2000 or 3000 and rerun.
SUBSET_SIZE = None
BATCH_SIZE = 16
EPSILON_VALUES = [0.03]
PGD_STEPS = 20
TARGET_MODE = "max_dist"
SAVE_SAMPLE_COUNT = 16

subset_label = "full" if SUBSET_SIZE is None else f"subset{SUBSET_SIZE}"
eps_label = "_".join(str(e).replace(".", "p") for e in EPSILON_VALUES)
RUN_NAME = f"caltech101_{subset_label}_eps{eps_label}_pgd{PGD_STEPS}_{TARGET_MODE}_seed{SEED}"
RUN_DIR = Path(RESULTS_ROOT) / "general_cloaking_feature_shift_full_final" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
SAMPLE_DIR = RUN_DIR / "samples"
FIGURE_DIR = RUN_DIR / "figures"
TENSOR_DIR = RUN_DIR / "tensors"
for d in [TABLE_DIR, SAMPLE_DIR, FIGURE_DIR, TENSOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("RUN_DIR:", RUN_DIR)
print("SUBSET_SIZE:", SUBSET_SIZE)
print("EPSILON_VALUES:", EPSILON_VALUES)
print("PGD_STEPS:", PGD_STEPS)


## 3. Reproducibility helpers


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)


## 4. Load Caltech-101 and surrogate feature extractor


In [ ]:
train_loader, _unused_loader, num_classes = get_caltech101(
    root=DATA_ROOT,
    img_size=IMG_SIZE,
    subset_size=SUBSET_SIZE,
    batch_size=BATCH_SIZE,
    train_ratio=1.0,
    seed=SEED,
    download=True,
    num_workers=2,
)

num_images = len(train_loader.dataset)
print("num_classes:", num_classes)
print("num_images:", num_images)
print("batches:", len(train_loader))

surrogate = get_surrogate_resnet50(DEVICE)
surrogate.eval()
print("surrogate: ResNet-50 ImageNet feature extractor")


## 5. Feature-space Cloaking benchmark


In [ ]:
def feature_metrics(surrogate, x_orig, x_protected, target_features):
    with torch.no_grad():
        f_orig = F.normalize(surrogate(x_orig).float(), dim=1)
        f_prot = F.normalize(surrogate(x_protected).float(), dim=1)
        target = F.normalize(target_features.float(), dim=1)

        cos_orig_prot = F.cosine_similarity(f_orig, f_prot, dim=1)
        l2_shift = torch.norm(f_prot - f_orig, p=2, dim=1)
        target_before = F.cosine_similarity(f_orig, target, dim=1)
        target_after = F.cosine_similarity(f_prot, target, dim=1)
        target_gain = target_after - target_before

    return {
        "feature_cosine_orig_protected": cos_orig_prot.detach().cpu(),
        "feature_l2_shift": l2_shift.detach().cpu(),
        "target_cosine_before": target_before.detach().cpu(),
        "target_cosine_after": target_after.detach().cpu(),
        "target_cosine_gain": target_gain.detach().cpu(),
    }


def save_samples(x_orig, x_protected, epsilon, max_samples=12):
    eps_name = str(epsilon).replace('.', 'p')
    eps_dir = SAMPLE_DIR / f"eps{eps_name}"
    eps_dir.mkdir(parents=True, exist_ok=True)
    n = min(max_samples, x_orig.size(0))
    for i in range(n):
        tensor_to_image(x_orig[i].cpu()).save(eps_dir / f"original_{i:03d}.png")
        tensor_to_image(x_protected[i].cpu()).save(eps_dir / f"protected_{i:03d}.png")
        noise = ((x_protected[i].detach().cpu() - x_orig[i].detach().cpu()) * 10 + 0.5).clamp(0, 1)
        tensor_to_image(noise).save(eps_dir / f"noise_x10_{i:03d}.png")

    fig, axes = plt.subplots(n, 3, figsize=(8, max(2, 2.2 * n)))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)
    for i in range(n):
        orig = x_orig[i].detach().cpu().permute(1, 2, 0).numpy()
        prot = x_protected[i].detach().cpu().permute(1, 2, 0).numpy()
        noise = ((x_protected[i].detach().cpu() - x_orig[i].detach().cpu()) * 10 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[i, 0].imshow(orig)
        axes[i, 1].imshow(prot)
        axes[i, 2].imshow(noise)
        axes[i, 0].set_title("Original")
        axes[i, 1].set_title("Cloaked")
        axes[i, 2].set_title("Noise x10")
        for ax in axes[i]:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"before_after_noise_eps{eps_name}.png", dpi=160)
    plt.close(fig)


all_summary = []

for epsilon in EPSILON_VALUES:
    print("=" * 80)
    print(f"Cloaking epsilon={epsilon}")
    start_time = time.perf_counter()

    per_batch_rows = []
    first_orig = None
    first_protected = None

    for batch_idx, (x, _y) in enumerate(tqdm(train_loader, desc=f"eps={epsilon}")):
        x = x.to(DEVICE)
        target_features = select_cloak_target(surrogate, x, mode=TARGET_MODE, device=DEVICE)
        x_protected = cloak_images(
            surrogate,
            x,
            epsilon=epsilon,
            pgd_steps=PGD_STEPS,
            pgd_alpha=max(epsilon / 10, 1 / 255),
            target_mode=TARGET_MODE,
            target_features=target_features,
            device=DEVICE,
        )

        feat = feature_metrics(surrogate, x, x_protected, target_features)
        row = {
            "batch_idx": batch_idx,
            "batch_size": x.size(0),
            "epsilon": epsilon,
            "feature_cosine_orig_protected": round(float(feat["feature_cosine_orig_protected"].mean()), 4),
            "feature_l2_shift": round(float(feat["feature_l2_shift"].mean()), 4),
            "target_cosine_before": round(float(feat["target_cosine_before"].mean()), 4),
            "target_cosine_after": round(float(feat["target_cosine_after"].mean()), 4),
            "target_cosine_gain": round(float(feat["target_cosine_gain"].mean()), 4),
            "psnr": compute_psnr(x.detach().cpu(), x_protected.detach().cpu()),
            "ssim": compute_ssim(x.detach().cpu(), x_protected.detach().cpu()),
            "linf": compute_linf(x.detach().cpu(), x_protected.detach().cpu()),
        }
        per_batch_rows.append(row)

        if first_orig is None:
            first_orig = x.detach().cpu()
            first_protected = x_protected.detach().cpu()

        del x, x_protected, target_features
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    per_batch_df = pd.DataFrame(per_batch_rows)
    eps_name = str(epsilon).replace('.', 'p')
    per_batch_path = TABLE_DIR / f"cloaking_per_batch_eps{eps_name}.csv"
    per_batch_df.to_csv(per_batch_path, index=False)

    weighted = per_batch_df.copy()
    total = weighted["batch_size"].sum()
    summary = {
        "technique": "general_feature_space_cloaking",
        "dataset": "Caltech-101",
        "surrogate_model": "ResNet-50 ImageNet feature extractor",
        "subset_size": SUBSET_SIZE,
        "image_size": IMG_SIZE,
        "epsilon": epsilon,
        "pgd_steps": PGD_STEPS,
        "target_mode": TARGET_MODE,
        "feature_cosine_orig_protected": round(float((weighted["feature_cosine_orig_protected"] * weighted["batch_size"]).sum() / total), 4),
        "feature_l2_shift": round(float((weighted["feature_l2_shift"] * weighted["batch_size"]).sum() / total), 4),
        "target_cosine_before": round(float((weighted["target_cosine_before"] * weighted["batch_size"]).sum() / total), 4),
        "target_cosine_after": round(float((weighted["target_cosine_after"] * weighted["batch_size"]).sum() / total), 4),
        "target_cosine_gain": round(float((weighted["target_cosine_gain"] * weighted["batch_size"]).sum() / total), 4),
        "psnr": round(float((weighted["psnr"] * weighted["batch_size"]).sum() / total), 4),
        "ssim": round(float((weighted["ssim"] * weighted["batch_size"]).sum() / total), 4),
        "linf_max": round(float(weighted["linf"].max()), 4),
        "runtime_seconds": round(time.perf_counter() - start_time, 2),
        "run_name": RUN_NAME,
        "run_dir": str(RUN_DIR),
    }
    all_summary.append(summary)

    save_samples(first_orig, first_protected, epsilon, max_samples=SAVE_SAMPLE_COUNT)
    torch.save({"x_orig": first_orig, "x_protected": first_protected}, TENSOR_DIR / f"sample_batch_eps{eps_name}.pt")
    print(summary)

summary_df = pd.DataFrame(all_summary)
summary_path = TABLE_DIR / "general_cloaking_feature_shift_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)
summary_df


## 6. Plot epsilon trade-off


In [ ]:
summary_df = pd.read_csv(TABLE_DIR / "general_cloaking_feature_shift_summary.csv")

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(summary_df["epsilon"], summary_df["feature_cosine_orig_protected"], marker="o", label="cosine(original, protected)")
ax1.plot(summary_df["epsilon"], summary_df["target_cosine_gain"], marker="o", label="target cosine gain")
ax1.set_xlabel("epsilon")
ax1.set_ylabel("feature metric")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(summary_df["epsilon"], summary_df["psnr"], marker="s", color="green", label="PSNR")
ax2.plot(summary_df["epsilon"], summary_df["ssim"], marker="s", color="purple", label="SSIM")
ax2.set_ylabel("image quality")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "cloaking_epsilon_tradeoff.png", dpi=160)
plt.show()
print("Saved:", FIGURE_DIR / "cloaking_epsilon_tradeoff.png")


## 7. Interpretation notes


In [ ]:
print("Use feature metrics as the main Cloaking evidence:")
print("- Lower cosine(original, protected) means stronger feature disruption.")
print("- Higher target cosine gain means protected embeddings move toward selected target features.")
print("- PSNR/SSIM/Linf show visual quality and perturbation bound.")
print("Do not use classifier clean accuracy as the main Cloaking metric in this notebook.")
print("RUN_DIR:", RUN_DIR)
